In [11]:
from pathlib import Path
import re
from typing import Iterable

# Data manipulation and cleaning helpers
import pandas as pd
from bs4 import BeautifulSoup
from html import unescape
from unidecode import unidecode

# Tokenization, stop words, and stemming
import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Vectorization utilities for building the preprocessing pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer

In [12]:
#  Locate CSV files that contain the training texts and labels
DATA_DIR = Path('../../Dataset')
X_train_path = DATA_DIR / 'X_train.csv'
Y_train_path = DATA_DIR / 'Y_train.csv'

# Read both datasets while preserving the shared index for later merging
X_train = pd.read_csv(X_train_path, index_col=0)
Y_train = pd.read_csv(Y_train_path, index_col=0)

In [14]:
# fusion et préparation des colonnes textuelles — consolide les champs 
# utiles pour que chaque produit soit représenté par un texte exploitable

# Combine features and labels, then assemble a single raw text column
train_df = X_train.join(Y_train, how='left')
text_columns = ['designation', 'description']

# Replace missing descriptions/titles with blanks to avoid NaN issues downstream
train_df[text_columns] = train_df[text_columns].fillna('')

# Concatenate the textual fields into one string per product
train_df['text_raw'] = train_df[text_columns].agg(' '.join, axis=1)

train_df.head()

,designation,description,productid,imageid,prdtypecode,text_raw
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,,3804725264,1263597046,10,Olivia: Personalisiertes Notizbuch / 150 Seite...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,,436067568,1008141237,2280,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,Grand Stylet Ergonomique Bleu Gamepad Nintendo...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,,50418756,457047496,1280,Peluche Donald - Europe - Disneyland 2000 (Mar...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,La Guerre Des Tuques Luc a des id&eacute;es de...


In [5]:
# fonctions de nettoyage — centralise les routines qui réduisent le bruit lexical avant la vectorisation

# Helper functions to strip HTML, normalize whitespace, and standardize text
def strip_html(text: str) -> str:
    """Remove HTML tags while keeping readable spacing."""
    if not text:
        return ''
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')


def normalize_whitespace(text: str) -> str:
    """Collapse multiple spaces/newlines into a single space."""
    return re.sub(r'\s+', ' ', text).strip()


def clean_text(text: str) -> str:
    """Full cleaning pipeline applied before vectorization."""
    if text is None:
        text = ''
    text = unescape(text)  # Decode HTML entities like &eacute;
    text = strip_html(text)
    text = text.lower()
    text = unidecode(text)  # Remove accents to harmonize tokens
    text = normalize_whitespace(text)
    return text

In [15]:
# tokenisation et racinisation — prépare le découpage et la normalisation
# qui alimentent les vecteurs de caractéristiques

# Download tokenization resources on first run and prepare stop-word/stemmer tools
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Build a bilingual stop-word list to drop high-frequency words in French and English
french_stopwords = set(stopwords.words('french'))
english_stopwords = set(stopwords.words('english'))
stopword_set = french_stopwords | english_stopwords
stemmer = SnowballStemmer('french')


def tokenize_and_stem(text: str) -> Iterable[str]:
    """Tokenize cleaned text, filter noise, and reduce words to their stems."""
    tokens = wordpunct_tokenize(text)
    filtered = [tok for tok in tokens if tok.isalpha()]
    filtered = [tok for tok in filtered if tok not in stopword_set and len(tok) > 2]
    stemmed = [stemmer.stem(tok) for tok in filtered]
    return stemmed


In [17]:
# configuration des vectoriseurs — construit les transformateurs TF-IDF qui convertiront 
# le texte nettoyé en caractéristiques numériques exploitables

# Word-level TF-IDF with stemming-aware tokenizer captures vocabulary and bigrams
word_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    tokenizer=tokenize_and_stem,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
)

# Character n-gram TF-IDF to capture subword patterns and handle misspellings
char_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=5,
    max_df=0.9,
)

# Combine word and character features into a single sparse representation
text_vectorizer = FeatureUnion([
    ('word', word_vectorizer),
    ('char', char_vectorizer),
])

# Wrap the feature union in a pipeline that selects the raw text column
text_pipeline = Pipeline([
    ('select_text', FunctionTransformer(lambda df: df['text_raw'], validate=False)),
    ('vectorize', text_vectorizer),
])

In [ ]:
# éparation et vectorisation des jeux d'entraînement/validation — produit 
# les matrices finales tout en préservant une validation représentative pour l'évaluation

# Separate features/target and create a stratified train/validation split
X_features = train_df[['text_raw']].copy()
y_target = train_df['prdtypecode'].copy()

X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target,
)

# Fit the text pipeline on the training data and transform both splits
vectorizer_model = text_pipeline.fit(X_train_split, y_train_split)

X_train_vectors = vectorizer_model.transform(X_train_split)
X_valid_vectors = vectorizer_model.transform(X_valid_split)

# Display the resulting sparse matrix shapes for sanity checking
print(f'Train matrix shape: {X_train_vectors.shape}')
print(f'Validation matrix shape: {X_valid_vectors.shape}')

Train matrix shape: (67932, 316675)
Validation matrix shape: (16984, 316675)
